# Implementing and Checking a Binary Heap


```{contents}
:local:
:depth: 2
```


```{index} Repair Only the Part an Operation Can Disturb
```



```{index} Repair Only the Part an Operation Can Disturb
```

## Repair Only the Part an Operation Can Disturb



Enqueue adds a new entry at the next open leaf position so the complete shape is preserved. All old parent-child relations are unchanged. Only the new entry's path toward the root can violate min-heap order. Bubble-up compares with the parent and swaps while the child is smaller. Moving upward strictly decreases its index, so repair terminates.

Start with [2, 5, 4] and enqueue 1. Appending produces [2, 5, 4, 1]. Comparing 1 with parent 5 gives [2, 1, 4, 5]. Comparing that 1 with root 2 gives [1, 2, 4, 5]. Do not sort the whole array; that would enforce a stronger and more expensive condition than the operation needs.

Dequeue first saves the root result. Move the last live entry to the root and reduce Count. The only possible ordering defects now lie downward from that replacement. Choose the smaller existing child before swapping; exchanging with an arbitrary child can leave a smaller sibling above a larger parent relation.

| Heap after removing 1 | Inspection | Next step |
| --- | --- | --- |
| [5, 2, 4] | Children 2 and 4 | Choose 2 |
| [2, 5, 4] | 5 has no child | Stop |

A singleton removal needs no child access after Count becomes zero. An empty removal must follow the stated failure policy without first reading position zero. Also clear unused reference slots when the implementation owns a backing array and should release references to removed objects. Reducing Count hides a slot from the abstract sequence, but it does not itself erase a stored reference.

The height of a complete heap is logarithmic in its live size, which bounds the number of repair levels. Array growth can add copying to an enqueue, so qualify an operation's cost rather than ignoring representation maintenance. Bulk building by bottom-up repair is a related algorithm with a different aggregate argument; it is not proved merely by multiplying one operation's bound.

Guided exercise: repeat enqueue with a value 6 that already belongs below its parent. Predict how many swaps occur and distinguish swaps from comparisons. Then remove down to empty and enqueue again. Those cases test both the invariant and whether endpoint handling leaves the structure reusable.

For related explanation, see the further reading below.[^development-reading]


```{index} Enqueue and Bubble Up
```

## Enqueue and Bubble Up




```{index} heap; bubble up
```


To insert into a heap, place the new item at the end to preserve the complete-tree shape, then **bubble up** while it violates the heap order rule.


In [1]:
using System;
using System.Collections.Generic;

var heap = new List<int> { 1, 3, 2, 7, 5, 6, 4 };
heap.Add(0);

int child = heap.Count - 1;
while (child > 0)
{
    int parent = (child - 1) / 2;
    if (heap[parent] <= heap[child])
    {
        break;
    }

    (heap[parent], heap[child]) = (heap[child], heap[parent]);
    child = parent;
}

Console.WriteLine(string.Join(", ", heap));


0, 1, 2, 3, 5, 6, 4, 7


The new value `0` starts at the end. It swaps with larger parents until the min-heap property is restored.


**Common mistake.** Computing the parent as `child / 2` instead of `(child - 1) / 2`. For index 1 both give 0, which hides the bug, but for index 2 the wrong formula gives 1 instead of 0, and the new value is compared with a sibling rather than its parent.


Trace the appended zero from index seven to the root:

| Stage | Index path | Array |
| --- | --- | --- |
| Append | 7 | 1, 3, 2, 7, 5, 6, 4, 0 |
| Swap with 7 | 7 → 3 | 1, 3, 2, 0, 5, 6, 4, 7 |
| Swap with 3 | 3 → 1 | 1, 0, 2, 3, 5, 6, 4, 7 |
| Swap with 1 | 1 → 0 | 0, 1, 2, 3, 5, 6, 4, 7 |

Before insertion, every old parent-child relation satisfies the order rule. Appending adds only one possible violation between the new item and its parent. Each swap repairs that relation and moves the possible violation upward. Stop at the root or when the parent is no larger. This explains why repairing one path is sufficient.

```{index} Dequeue and Sink Down
```

## Dequeue and Sink Down




```{index} heap; sink down
```


To remove the minimum item, remove the root, move the last item to the root, then **sink down** until the heap order rule is restored.


In [1]:
using System;
using System.Collections.Generic;

var heap = new List<int> { 1, 3, 2, 7, 5, 6, 4 };
int removed = heap[0];
heap[0] = heap[^1];
heap.RemoveAt(heap.Count - 1);

int parent = 0;
while (true)
{
    int left = 2 * parent + 1;
    int right = 2 * parent + 2;
    int smallest = parent;

    if (left < heap.Count && heap[left] < heap[smallest]) smallest = left;
    if (right < heap.Count && heap[right] < heap[smallest]) smallest = right;
    if (smallest == parent) break;

    (heap[parent], heap[smallest]) = (heap[smallest], heap[parent]);
    parent = smallest;
}

Console.WriteLine($"Removed: {removed}");
Console.WriteLine(string.Join(", ", heap));


Removed: 1
2, 3, 4, 7, 5, 6


Both bubble-up and sink-down follow one root-to-leaf path, so their work grows with heap height, not with every item in the heap.


**Common mistake.** Swapping with the left child whenever it is smaller than the parent, without checking the right child. If the right child is smaller still, the left child ends up above a smaller sibling and the order rule is broken one level down.


In the removal example, replacing root 1 with the last value 4 gives `4, 3, 2, 7, 5, 6`. Compare both children before choosing the swap: 2 is smaller than 3, so swap indexes zero and two. The result is `2, 3, 4, 7, 5, 6`. The next child is 6; 4 is already no larger, so stop.

Choosing a larger child can leave the root larger than the other child. Sink-down must choose the smaller existing child and handle a last parent with only a left child. Removing the last array position preserves the complete shape. For a singleton, removal leaves no root to repair.

The complete generic heap below hides the backing list and accepts an optional comparer. The element type must support the default ordering, or you must supply a comparer. `Peek` and `Dequeue` throw on empty; `TryDequeue` distinguishes empty from a stored default value. Equal values are legal, but this heap does not impose stable arrival order on equal comparisons.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

var priorities = new BinaryMinHeap<int>();
foreach (int urgency in new[] { 3, 0, 2, 0, 5 }) priorities.Enqueue(urgency);
Console.WriteLine($"count={priorities.Count}, minimum={priorities.Peek()}, valid={priorities.IsValid()}");
Console.WriteLine("raw=" + string.Join(", ", priorities.Snapshot()));
while (priorities.TryDequeue(out int urgency)) Console.WriteLine(urgency);
priorities.Enqueue(0);
Console.WriteLine($"reused zero={priorities.Dequeue()}, empty={priorities.Count == 0}");

public class BinaryMinHeap<T>
{
    private readonly List<T> items = new();
    private readonly IComparer<T> comparer;
    public int Count => items.Count;
    public int Capacity => items.Capacity;
    public BinaryMinHeap(IComparer<T>? comparer = null)
        => this.comparer = comparer ?? Comparer<T>.Default;

    public void Enqueue(T item)
    {
        items.Add(item);
        int child = items.Count - 1;
        while (child > 0)
        {
            int parent = (child - 1) / 2;
            if (comparer.Compare(items[parent], items[child]) <= 0) break;
            (items[parent], items[child]) = (items[child], items[parent]);
            child = parent;
        }
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Heap is empty.");
        return items[0];
    }

    public T Dequeue()
    {
        T minimum = Peek();
        T last = items[^1];
        items.RemoveAt(items.Count - 1);
        if (items.Count == 0) return minimum;
        items[0] = last;
        int parent = 0;
        while (parent < items.Count / 2)
        {
            int left = 2 * parent + 1;
            int right = left + 1;
            int smaller = left;
            if (right < items.Count && comparer.Compare(items[right], items[left]) < 0)
                smaller = right;
            if (comparer.Compare(items[parent], items[smaller]) <= 0) break;
            (items[parent], items[smaller]) = (items[smaller], items[parent]);
            parent = smaller;
        }
        return minimum;
    }

    public bool TryDequeue(out T item)
    {
        if (items.Count == 0) { item = default!; return false; }
        item = Dequeue();
        return true;
    }

    public bool IsValid()
    {
        for (int child = 1; child < items.Count; child++)
            if (comparer.Compare(items[(child - 1) / 2], items[child]) > 0) return false;
        return true;
    }

    public T[] Snapshot() => items.ToArray();
}

count=5, minimum=0, valid=True
raw=0, 0, 2, 3, 5
0
0
2
3
5
reused zero=0, empty=True


`List<T>` grows its storage when necessary. Bubble-up uses O(log n) comparisons, but a single enqueue that copies a full backing array can take O(n) time. Across a long sequence, storage growth contributes amortized O(1) per append; enqueue therefore has amortized O(log(n + 1)) time. Dequeue repairs at most one downward path and takes O(log(n + 1)); it does not shrink this backing list. Peek and Count take O(1). The iterative repair uses O(1) extra space, while the stored items use O(n) space.

`IsValid` scans all parent-child relations, so it takes Θ(n) time. It is a diagnostic method, not part of every production operation. `Snapshot` also takes Θ(n) time and space to copy the occupied values. Checking the minimum alone would miss a violation inside a subtree. Test empty and singleton states, duplicate priorities, an odd final child, ascending/descending input, draining, and reuse.

A heap can also be built from an existing array by sinking internal nodes from right to left. Most nodes are near leaves and require little repair; summing that work gives O(n) bottom-up construction. Repeated insertion has an O(n log n) worst-case bound. This chapter implements repeated enqueue first; bottom-up construction is an extension.

```{index} Practice: Check Every Parent
```

## Practice: Check Every Parent




```{index} Practice: Check Every Parent
```


Implement the invariant check for all non-root positions, using the heap's configured comparer.


In [ ]:
using System;
using System.Collections.Generic;

var heap=new BinaryMinHeap<int>();foreach(int value in new[]{4,1,3,1})heap.Enqueue(value);
while(heap.TryDequeue(out int next))Console.WriteLine(next);

public class BinaryMinHeap<T>
{
    private readonly List<T> items = new List<T>();
    private readonly IComparer<T> comparer;
    public int Count => items.Count;
    public int Capacity => items.Capacity;
    public BinaryMinHeap(IComparer<T>? comparer = null)
        => this.comparer = comparer ?? Comparer<T>.Default;

    public void Enqueue(T item)
    {
        items.Add(item);
        int child = items.Count - 1;
        while (child > 0)
        {
            int parent = (child - 1) / 2;
            if (comparer.Compare(items[parent], items[child]) <= 0) break;
            (items[parent], items[child]) = (items[child], items[parent]);
            child = parent;
        }
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Heap is empty.");
        return items[0];
    }

    public T Dequeue()
    {
        T minimum = Peek();
        T last = items[^1];
        items.RemoveAt(items.Count - 1);
        if (items.Count == 0) return minimum;
        items[0] = last;
        int parent = 0;
        while (parent < items.Count / 2)
        {
            int left = 2 * parent + 1;
            int right = left + 1;
            int smaller = left;
            if (right < items.Count && comparer.Compare(items[right], items[left]) < 0)
                smaller = right;
            if (comparer.Compare(items[parent], items[smaller]) <= 0) break;
            (items[parent], items[smaller]) = (items[smaller], items[parent]);
            parent = smaller;
        }
        return minimum;
    }

    public bool TryDequeue(out T item)
    {
        if (items.Count == 0) { item = default!; return false; }
        item = Dequeue();
        return true;
    }

    public bool IsValid()
    {
        for (int child = 1; child < items.Count; child++)
            { /* TODO: reject a child that precedes its parent. */ }
        return true;
    }

    public T[] Snapshot() => items.ToArray();
}


In [1]:
using System;
using System.Collections.Generic;

var heap=new BinaryMinHeap<int>();foreach(int value in new[]{4,1,3,1})heap.Enqueue(value);
while(heap.TryDequeue(out int next))Console.WriteLine(next);

public class BinaryMinHeap<T>
{
    private readonly List<T> items = new List<T>();
    private readonly IComparer<T> comparer;
    public int Count => items.Count;
    public int Capacity => items.Capacity;
    public BinaryMinHeap(IComparer<T>? comparer = null)
        => this.comparer = comparer ?? Comparer<T>.Default;

    public void Enqueue(T item)
    {
        items.Add(item);
        int child = items.Count - 1;
        while (child > 0)
        {
            int parent = (child - 1) / 2;
            if (comparer.Compare(items[parent], items[child]) <= 0) break;
            (items[parent], items[child]) = (items[child], items[parent]);
            child = parent;
        }
    }

    public T Peek()
    {
        if (items.Count == 0) throw new InvalidOperationException("Heap is empty.");
        return items[0];
    }

    public T Dequeue()
    {
        T minimum = Peek();
        T last = items[^1];
        items.RemoveAt(items.Count - 1);
        if (items.Count == 0) return minimum;
        items[0] = last;
        int parent = 0;
        while (parent < items.Count / 2)
        {
            int left = 2 * parent + 1;
            int right = left + 1;
            int smaller = left;
            if (right < items.Count && comparer.Compare(items[right], items[left]) < 0)
                smaller = right;
            if (comparer.Compare(items[parent], items[smaller]) <= 0) break;
            (items[parent], items[smaller]) = (items[smaller], items[parent]);
            parent = smaller;
        }
        return minimum;
    }

    public bool TryDequeue(out T item)
    {
        if (items.Count == 0) { item = default!; return false; }
        item = Dequeue();
        return true;
    }

    public bool IsValid()
    {
        for (int child = 1; child < items.Count; child++)
            if (comparer.Compare(items[(child - 1) / 2], items[child]) > 0) return false;
        return true;
    }

    public T[] Snapshot() => items.ToArray();
}


1
1
3
4


The output is `1`, `1`, `3`, `4`. Both copies of 1 come out, in some order, before 3 and 4; duplicates are ordinary values to a heap. The invariant check runs against the same comparer the heap uses, so a heap built with a reversed comparer would validate under its own rule, not under `<=`.


```{rubric} Footnotes
```


[^1802_heap_implementation-1]: The teaching implementation is a binary min-heap. A library priority queue may use a different internal layout while honoring its public priority contract.


[^development-reading]: Further reading: [OpenDSA, Heaps and Priority Queues](https://opendsax.cs.vt.edu/ODSA/Books/Everything/html/Heaps.html). The worked business scenarios and guided reasoning here are original teaching examples.
